In [1]:
import sys
import os
import numpy as np
import scipy.linalg as la
from scipy.signal import cont2discrete
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter, FFMpegWriter
from mpl_toolkits.mplot3d import Axes3D 
import matplotlib as mpl

# Add project root to sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.append(project_root)

import utils.config as cfg
from utils.quadcopter_model import quad_hover_linear_model, create_discrete_model, gaussian_dist, visualize_gaussian_samples, animate_trajectories, plot_angles, plot_xyz, plot_controls_grid, plot_angles_mean, plot_xyz_mean, plot_controls_grid_mean, summarize_checks
from utils.simple_MLP import PolicyMLP, rollout_policy, traj_cost, train_nn_policy, evaluate_policy_cost, plot_loss_evolution

# Plotting parameters
plt.style.use('default')
plt.rcParams['figure.dpi'] = 400

# Enable LaTeX rendering
mpl.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"], 
    "axes.unicode_minus": False 
})

# Consistent printing of numpy arrays
np.set_printoptions(precision=3, suppress=True)

In [2]:
# Create and discretize model
A, B, C, D = quad_hover_linear_model()
# print("A =\n", A, "\n\nB =\n", B, "\n\nC =\n", C, "\n\nD =\n", D)

A_d, B_tilde_d, C_d, D_d = create_discrete_model(A, B, C, D)
B_d = B_tilde_d[:, :4]
# print("A_d =\n", A_d, "\n\nB_tilde_d =\n", B_tilde_d, "\n\nC_d =\n", C_d, "\n\nD_d =\n", D_d)

In [ ]:
# Sanity check
summarize_checks("Hover (CT)", A, B, C, discrete=False)
summarize_checks("Hover (DT)", A_d, B_d, C_d, discrete=True)

# CT system is marginally stable (not asymptotically stable): linearized around hover, 
# so positions and angles have integrator dynamics (they don’t decay by themselves).
# DT: discretizing integrators, we get poles at z=1, so still marginally stable.

In [14]:
# Training points

# Generate disturbance wind samples for training
train_dist_mean_x_1  = 5.0
train_dist_sigma_x_1 = 1.0
nb_training_points_1 = 500
dist_x_train_1 = gaussian_dist(train_dist_mean_x_1, train_dist_sigma_x_1, nb_training_points_1, device="cpu")           # x-axis disturbance (random wind per trajectory, constant over time)

train_dist_mean_x_2  = -5.0
train_dist_sigma_x_2 = 1.0
nb_training_points_2 = 5
dist_x_train_2 = gaussian_dist(train_dist_mean_x_2, train_dist_sigma_x_2, nb_training_points_2, device="cpu")           # x-axis disturbance (random wind per trajectory, constant over time)

nb_training_points = nb_training_points_1 +  nb_training_points_2

dist_x_train = torch.cat([dist_x_train_1, dist_x_train_2], dim=0)
dist_y_train = torch.zeros_like(dist_x_train)                                                                   # y-axis disturbance (zero)
dist_z_train = -cfg.G * torch.ones_like(dist_x_train)                                                           # z-axis disturbance (gravity)
disturbances_train = torch.cat([dist_x_train, dist_y_train, dist_z_train], dim=-1)                              # [B, T, 3] # [B, T, 3] (B = number of trajectories (batch points), T = number of timesteps per trajectory, 3 = scalar disturbance value for that axis)

visualize_gaussian_samples(dist_x_train_1, train_dist_mean_x_1, train_dist_sigma_x_1, nb_training_points_1, "results/exp3_training_dist_samples_1.png")
visualize_gaussian_samples(dist_x_train_2, train_dist_mean_x_2, train_dist_sigma_x_2, nb_training_points_2, "results/exp3_training_dist_samples_2.png")

# Initial state conditions for training
x0_train_np = np.array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]*nb_training_points)
x0_train = torch.from_numpy(x0_train_np).float()

In [15]:
# TRAINING
device = "cpu"
epochs = 1000
compute_mean_every = 50
lr = 5e-3

policy, train_means, eval_means = train_nn_policy(
    A_d_nt=A_d, B_tilde_d_nt=B_tilde_d,
    init_points=x0_train, disturbances=disturbances_train,
    lr=lr, epochs=epochs,
    device=device, verbose_every=compute_mean_every, shuffle_each_epoch=True
)

[ep 50/1000]:   train_loss=2.5551e+06 ± 1.36e+06   |   eval_loss=7.1664e+05 ± 5.46e+05   |   |x_T|_avg=107.443
[ep 100/1000]:   train_loss=6.7682e+05 ± 3.67e+05   |   eval_loss=4.1572e+05 ± 4.10e+05   |   |x_T|_avg=96.015
[ep 150/1000]:   train_loss=2.5986e+04 ± 7.24e+03   |   eval_loss=1.4817e+04 ± 1.07e+04   |   |x_T|_avg=12.108
[ep 200/1000]:   train_loss=8.5365e+02 ± 2.30e+02   |   eval_loss=9.9882e+02 ± 1.38e+03   |   |x_T|_avg=0.978
[ep 250/1000]:   train_loss=7.9013e+02 ± 8.15e+02   |   eval_loss=3.9033e+02 ± 4.90e+02   |   |x_T|_avg=0.404
[ep 300/1000]:   train_loss=3.1365e+02 ± 4.44e+01   |   eval_loss=3.3225e+02 ± 3.64e+02   |   |x_T|_avg=0.348
[ep 350/1000]:   train_loss=3.1547e+02 ± 3.16e+01   |   eval_loss=3.1176e+02 ± 3.10e+02   |   |x_T|_avg=0.237
[ep 400/1000]:   train_loss=2.7557e+02 ± 5.71e+01   |   eval_loss=2.9658e+02 ± 2.77e+02   |   |x_T|_avg=0.208
[ep 450/1000]:   train_loss=4.1584e+02 ± 2.44e+02   |   eval_loss=3.0573e+02 ± 2.95e+02   |   |x_T|_avg=0.196
[ep 500

In [16]:
# Plot loss evolution
plot_loss_evolution(train_means, eval_means, epochs, compute_mean_every, filename="results/exp3_losses.png")

In [22]:
# TESTING on training points
with torch.no_grad():
    X_training, U_training, mean_cost_training, std_cost_training, costs_training = evaluate_policy_cost(A_d, B_tilde_d, policy, x0_train, disturbances_train, device=device)

print(f"Training cost:  {mean_cost_training:.3e} ± {std_cost_training:.2e}")

plot_xyz_mean(X_training, filename="results/exp3_training_xyz.png")
plot_angles_mean(X_training, filename="results/exp3_training_angles.png")
plot_controls_grid_mean(U_training, filename="results/exp3_training_controls.png")
animate_trajectories(X_training, "results/exp3_training_xyz_3d.mp4")

Training cost:  2.788e+02 ± 2.53e+02
Frame 1/200
Frame 5/200
Frame 9/200
Frame 13/200
Frame 17/200
Frame 21/200
Frame 25/200
Frame 29/200
Frame 33/200
Frame 37/200
Frame 41/200
Frame 45/200
Frame 49/200
Frame 53/200
Frame 57/200
Frame 61/200
Frame 65/200
Frame 69/200
Frame 73/200
Frame 77/200
Frame 81/200
Frame 85/200
Frame 89/200
Frame 93/200
Frame 97/200
Frame 101/200
Frame 105/200
Frame 109/200
Frame 113/200
Frame 117/200
Frame 121/200
Frame 125/200
Frame 129/200
Frame 133/200
Frame 137/200
Frame 141/200
Frame 145/200
Frame 149/200
Frame 153/200
Frame 157/200
Frame 161/200
Frame 165/200
Frame 169/200
Frame 173/200
Frame 177/200
Frame 181/200
Frame 185/200
Frame 189/200
Frame 193/200
Frame 197/200


In [25]:
# Testing points

# Generate disturbance wind samples for testing
test_dist_mean_x  = 5.0
test_dist_sigma_x = 1.0
nb_testing_points  = 500

dist_x_test = gaussian_dist(test_dist_mean_x, test_dist_sigma_x, nb_testing_points, device="cpu")     # x-axis disturbance (random wind per trajectory, constant over time)
dist_y_test = torch.zeros_like(dist_x_test)                                                           # y-axis disturbance (zero)
dist_z_test = -cfg.G * torch.ones_like(dist_x_test)                                                   # z-axis disturbance (gravity)
disturbances_test = torch.cat([dist_x_test, dist_y_test, dist_z_test], dim=-1)                        # [B, T, 3] # [B, T, 3] (B = number of trajectories (batch points), T = number of timesteps per trajectory, 3 = scalar disturbance value for that axis)

visualize_gaussian_samples(dist_x_test, test_dist_mean_x, test_dist_sigma_x, nb_testing_points, "results/exp3_testing_dist_samples.png")

# Initial state conditions for testing
x0_test_np = np.array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]*nb_testing_points)
x0_test = torch.from_numpy(x0_test_np).float()

In [26]:
# TESTING on testing points
with torch.no_grad():
    X_testing, U_testing, mean_cost_testing, std_cost_testing, costs_testing = evaluate_policy_cost(A_d, B_tilde_d, policy, x0_test, disturbances_test, device=device)

print(f"Testing cost:  {mean_cost_testing:.3e} ± {std_cost_testing:.2e}")

plot_xyz_mean(X_testing, filename="results/exp3_testing_xyz_N1.png")
plot_angles_mean(X_testing, filename="results/exp3_testing_angles_N1.png")
plot_controls_grid_mean(U_testing, filename="results/exp3_testing_controls_N1.png")
animate_trajectories(X_testing, "results/exp3_testing_xyz_3d_N1.mp4")

Testing cost:  2.875e+02 ± 2.68e+02
Frame 1/200
Frame 5/200
Frame 9/200
Frame 13/200
Frame 17/200
Frame 21/200
Frame 25/200
Frame 29/200
Frame 33/200
Frame 37/200
Frame 41/200
Frame 45/200
Frame 49/200
Frame 53/200
Frame 57/200
Frame 61/200
Frame 65/200
Frame 69/200
Frame 73/200
Frame 77/200
Frame 81/200
Frame 85/200
Frame 89/200
Frame 93/200
Frame 97/200
Frame 101/200
Frame 105/200
Frame 109/200
Frame 113/200
Frame 117/200
Frame 121/200
Frame 125/200
Frame 129/200
Frame 133/200
Frame 137/200
Frame 141/200
Frame 145/200
Frame 149/200
Frame 153/200
Frame 157/200
Frame 161/200
Frame 165/200
Frame 169/200
Frame 173/200
Frame 177/200
Frame 181/200
Frame 185/200
Frame 189/200
Frame 193/200
Frame 197/200


In [24]:
# Cost comparison
pct_increase = (mean_cost_testing - mean_cost_training) / mean_cost_training * 100
print(f"Cost increase from training to testing: {pct_increase:.2f} %")

Cost increase from training to testing: 988.39 %
